# 01 - MP4 to Cropped Labelled Frames

This notebook reads Copernicus MP4 + timeline TXT pairs from `raw_to_be_processed/`, archives the raw files into `data/raw/<label>/<latitude>_<longitude>/`, extracts one frame per timeline date, crops 5% from every border, and saves processed PNG frames into `data/processed/<label>/<latitude>_<longitude>/`.

Input naming:

```text
raw_to_be_processed/
  3.5165528_101.9364861_high.mp4
  3.5165528_101.9364861.txt
  # also accepted: 3.5165528_101.9364861_high.txt
```

Timeline rule: line 1 maps to timestamp `0s`, line 2 maps to `1s`, and so on.

In [32]:
from pathlib import Path
import importlib.util
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

INBOX_DIR = PROJECT_ROOT / "raw_to_be_processed"
DATA_DIR = PROJECT_ROOT / "data"
CROP_PERCENT = 5.0
FORCE_REPROCESS = False

print("Project root:", PROJECT_ROOT)
print("Inbox:", INBOX_DIR)
print("Data dir:", DATA_DIR)
print("OpenCV available:", importlib.util.find_spec("cv2") is not None)

Project root: c:\Users\Aki\Downloads\AML - Final Project\code
Inbox: c:\Users\Aki\Downloads\AML - Final Project\code\raw_to_be_processed
Data dir: c:\Users\Aki\Downloads\AML - Final Project\code\data
OpenCV available: True


## Dependency Check

This pipeline needs `opencv-python` to decode MP4 frames. If OpenCV is not available, install it first:

```powershell
pip install -r requirements.txt
```

or:

```powershell
pip install opencv-python
```

In [33]:
if importlib.util.find_spec("cv2") is None:
    raise ImportError(
        "opencv-python is not installed. Run `pip install -r requirements.txt` "
        "from the project root, then restart this notebook kernel."
    )

## Preview Incoming Samples

In [34]:
from src.preprocessing.process_raw_videos import discover_samples, read_timeline

samples = discover_samples(INBOX_DIR)
preview_rows = []
for sample in samples:
    dates = read_timeline(sample.timeline_path)
    preview_rows.append({
        "sample_id": sample.sample_id,
        "label": sample.label,
        "video": sample.video_path.name,
        "timeline": sample.timeline_path.name,
        "frame_count": len(dates),
        "start_date": dates[0],
        "end_date": dates[-1],
    })

preview_rows

[{'sample_id': '2.831080_101.703818',
  'label': 'low',
  'video': '2.831080_101.703818_low.mp4',
  'timeline': '2.831080_101.703818_low.txt',
  'frame_count': 15,
  'start_date': '2015-12-31',
  'end_date': '2025-10-31'},
 {'sample_id': '3.5165528_101.9364861',
  'label': 'high',
  'video': '3.5165528_101.9364861_high.mp4',
  'timeline': '3.5165528_101.9364861_high.txt',
  'frame_count': 14,
  'start_date': '2016-06-08',
  'end_date': '2025-07-11'}]

## Process MP4 Videos

Expected output for each sample:

```text
data/raw/<label>/<latitude>_<longitude>/
  original_video.mp4
  timeline.txt

data/processed/<label>/<latitude>_<longitude>/
  frame_000__YYYY-MM-DD.png
  frame_001__YYYY-MM-DD.png
  frame_metadata.csv
  processing_metadata.json
```

In [35]:
from src.preprocessing.process_raw_videos import process_inbox

results = process_inbox(
    inbox_dir=INBOX_DIR,
    data_dir=DATA_DIR,
    crop_percent=CROP_PERCENT,
    force=FORCE_REPROCESS,
)

print(json.dumps(results, indent=2))

[
  {
    "latitude": 2.83108,
    "longitude": 101.703818,
    "label": "low",
    "sample_id": "2.831080_101.703818",
    "raw_dir": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\raw\\low\\2.831080_101.703818",
    "raw_video": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\raw\\low\\2.831080_101.703818\\2.831080_101.703818_low.mp4",
    "raw_timeline": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\raw\\low\\2.831080_101.703818\\2.831080_101.703818.txt",
    "processed_dir": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\low\\2.831080_101.703818",
    "frame_count": 15,
    "crop_percent": 5.0,
    "status": "processed",
    "processed_at": "2026-06-17T03:47:54.233269+00:00",
    "gee_features_csv": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\low\\2.831080_101.703818\\gee_features.csv"
  },
  {
    "latitude": 3.5165528,
    "longitude": 101.9364861,
    "label": "high",
    "sample_id": 

## Verify Processed Output

In [36]:
for result in results:
    processed_dir = Path(result["processed_dir"])
    frame_paths = sorted(processed_dir.glob("frame_*.png"))
    print(processed_dir)
    print("PNG frame count:", len(frame_paths))
    print("Frame metadata exists:", (processed_dir / "frame_metadata.csv").exists())
    print("Processing metadata exists:", (processed_dir / "processing_metadata.json").exists())
    print("First frames:", [path.name for path in frame_paths[:3]])
    print("Last frames:", [path.name for path in frame_paths[-3:]])

c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\low\2.831080_101.703818
PNG frame count: 15
Frame metadata exists: True
Processing metadata exists: True
First frames: ['frame_000__2015-12-31.png', 'frame_001__2016-03-30.png', 'frame_002__2016-06-08.png']
Last frames: ['frame_012__2023-06-27.png', 'frame_013__2024-03-18.png', 'frame_014__2025-10-31.png']
c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\high\3.5165528_101.9364861
PNG frame count: 14
Frame metadata exists: True
Processing metadata exists: True
First frames: ['frame_000__2016-06-08.png', 'frame_001__2019-09-01.png', 'frame_002__2020-01-24.png']
Last frames: ['frame_011__2024-06-11.png', 'frame_012__2024-09-24.png', 'frame_013__2025-07-11.png']
